In [1]:
import Pkg
Pkg.add([
        "CSV",
        "DataFrames",
        "Glob",
        "Makie",
        "CairoMakie"
        ])

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [2]:
import CSV
using DataFrames
using Glob
using Makie, CairoMakie
using Statistics

In [5]:
dfs = [CSV.read(f, DataFrame) for f in glob("*matches.csv", "tennis-data")];

49-element Vector{DataFrame}:
 179×16 DataFrame
 Row │ match_id           year   slam     match_num  player1              play ⋯
     │ String31           Int64  String7  Int64      String31             Stri ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ 2011-ausopen-1101   2011  ausopen       1101  Rafael Nadal         Marc ⋯
   2 │ 2011-ausopen-1103   2011  ausopen       1103  Bernard Tomic        Jere
   3 │ 2011-ausopen-1105   2011  ausopen       1105  John Isner           Flor
   4 │ 2011-ausopen-1108   2011  ausopen       1108  Donald Young         Mari
   5 │ 2011-ausopen-1112   2011  ausopen       1112  Juan Ignacio Chela   Mich ⋯
   6 │ 2011-ausopen-1113   2011  ausopen       1113  David Nalbandian     Lley
   7 │ 2011-ausopen-1114   2011  ausopen       1114  Richard Berankis     Mari
   8 │ 2011-ausopen-1115   2011  ausopen       1115  Michael Russell      Matt
   9 │ 2011-ausopen-1116   2011  ausopen       1116  Jarkko Nieminen     

In [64]:
players = vcat([[m.player1; m.player2] for m in dfs]...) |>
    skipmissing |>
    collect |>
    x -> lowercase.(x) |>
    unique |>
    sort |>
    enumerate |>
    x -> reverse.(x) |>
    Dict

Dict{String, Int64} with 1849 entries:
  "mona barthel"       => 1276
  "bjorn fratangelo"   => 284
  "ernests gulbis"     => 569
  "nick kyrgios"       => 1317
  "p. cuevas"          => 1377
  "astra sharma"       => 231
  "m. sakkari"         => 1155
  "t monteiro"         => 1639
  "t. monteiro"        => 1655
  "e bouchard"         => 502
  "regina kulikova"    => 1474
  "borna gojo"         => 293
  "yannick maden"      => 1809
  "kyrian jacquet"     => 994
  "v. zvonareva"       => 1742
  "a. dolgopolov"      => 60
  "alejandro tabilo"   => 125
  "ryan sweeting"      => 1507
  "a. bolt"            => 55
  "timea babos"        => 1694
  "vitalia diatchenko" => 1774
  "alberta brianti"    => 120
  "francoise abanda"   => 629
  "c. taberner"        => 339
  "maxime teixeira"    => 1241
  ⋮                    => ⋮

In [69]:
for m in dfs
    dropmissing!(m, [:player1, :player2])
    transform!(m, :player1 => (names -> getindex.(Ref(players), lowercase.(names))) => :player1id)
    transform!(m, :player2 => (names -> getindex.(Ref(players), lowercase.(names))) => :player2id)
end

In [73]:
for m in dfs
    filename = "$(m.year[1])-$(m.slam[1])-matches.csv"
    CSV.write(joinpath("clean-data",filename), m)
end

In [74]:
df = vcat(dfs...)

Row,match_id,year,slam,match_num,player1,player2,status,winner,event_name,round,court_name,court_id,player1id,player2id,nation1,nation2
,String31,Int64,String15,Any,String31,String31,String15?,Int64?,String15?,String15?,String31?,String1?,Int64,Int64,String3?,String3?
1,2011-ausopen-1101,2011,ausopen,1101,Rafael Nadal,Marcos Daniel,Retired,1,Men's Singles,Round 1,Rod Laver Arena,A,1467,1190,ESP,BRA
2,2011-ausopen-1103,2011,ausopen,1103,Bernard Tomic,Jeremy Chardy,Complete,1,Men's Singles,Round 1,Hisense Arena,B,278,854,AUS,FRA
3,2011-ausopen-1105,2011,ausopen,1105,John Isner,Florent Serra,Complete,1,Men's Singles,Round 1,Court 6,H,876,621,USA,FRA
4,2011-ausopen-1108,2011,ausopen,1108,Donald Young,Marin Cilic,Complete,2,Men's Singles,Round 1,Show Court 2,D,494,1212,USA,CRO
5,2011-ausopen-1112,2011,ausopen,1112,Juan Ignacio Chela,Michael Llodra,Complete,2,Men's Singles,Round 1,Show Court 2,D,885,1252,ARG,FRA
6,2011-ausopen-1113,2011,ausopen,1113,David Nalbandian,Lleyton Hewitt,Complete,1,Men's Singles,Round 1,Rod Laver Arena,A,471,1054,ARG,AUS
7,2011-ausopen-1114,2011,ausopen,1114,Richard Berankis,Marinko Matosevic,Complete,1,Men's Singles,Round 1,Court 6,H,1483,1216,LTU,AUS
8,2011-ausopen-1115,2011,ausopen,1115,Michael Russell,Matthew Ebden,Complete,1,Men's Singles,Round 1,Margaret Court Arena,C,1254,1233,USA,AUS
9,2011-ausopen-1116,2011,ausopen,1116,Jarkko Nieminen,David Ferrer,Complete,2,Men's Singles,Round 1,Show Court 3,E,838,468,FIN,ESP


In [90]:
players = combine(
    groupby(
        vcat(
            select(df, [:player1id => :id, :player1 => :player]),
            select(df, [:player2id => :id, :player2 => :player])
            ),
        :player
        ),
    [:id => first => :id, :player => unique => :player]
)
sort!(players, :id)

Row,player,id
,String31,Int64
1,A Anisimova,1
2,A Balazs,2
3,A Barty,3
4,A Bedene,4
5,A Blinkova,5
6,A Bogdan,6
7,A Bolsova,7
8,A Bolt,8
9,A Bublik,9


In [91]:
CSV.write(joinpath("clean-data", "players.csv"), players)

"clean-data/players.csv"